In [284]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from Bio.SeqUtils import molecular_weight as calculate_molecular_weight

import sys
sys.path.insert(1, '../../scripts/')
from utils.load_environmental_variables import *
from utils import parameters as params
from utils import machinery as mach

In [285]:
human_model = params.human_model.copy()
metabolic_machinery = sorted(mach.metabolic_machinery)
expression_machinery = sorted(mach.expression_machinery)
psim_me = params.psim_me

In [ ]:
all_machinery = sorted(set(metabolic_machinery+expression_machinery))

In [24]:
ids = pd.read_csv(local_data_path + 'raw/MANE.GRCh38.v0.9.summary.txt', sep = '\t')
ids.Ensembl_Gene = ids.Ensembl_Gene.apply(lambda x: x.split('.')[0])
id_map = dict(zip(ids.Ensembl_Gene.tolist(), ids.HGNC_ID.tolist()))

In [257]:
# load proteomics data
proteomics = pd.read_excel(local_data_path + 'raw/robinson_proteomics.xlsx', sheet_name = 4, 
                           skip_rows= list(range(3)))

# format
proteomics.columns = proteomics.iloc[1,:]
proteomics.columns = ['ENSG_ID_full', 'ENSG_ID'] + proteomics.columns.tolist()[2:]
proteomics.drop(index = list(range(3)), columns = ['ENSG_ID_full'], inplace = True)
proteomics.reset_index(inplace = True, drop = True)

# map to hgnc id
proteomics['HGNC_ID'] = proteomics.ENSG_ID.map(id_map) # do mapping
proteomics = proteomics.loc[proteomics['HGNC_ID'].dropna().index, :] # get rid of unmapped
proteomics.drop(columns = ['ENSG_ID'], inplace = True)

# more formatting (convert to float)
for col in proteomics.columns[:-1]:
    proteomics[col] = proteomics[col].astype(float)
    
    
# get MW of all proteins
avg_aa = np.mean([calculate_molecular_weight(aa, seq_type = 'protein') for aa in params.amino_acids])
def get_mw(protein):
    n_x = protein.count('X')
    protein = protein.replace('X', '')
    return calculate_molecular_weight(protein, seq_type='protein') + (n_x*avg_aa)
hgnc_to_mw = dict(zip(psim_me.HGNC_ID, psim_me.PROTEIN_SEQ.apply(lambda x: get_mw(x))))
mw_ = proteomics.HGNC_ID.map(hgnc_to_mw)

In [ ]:
# index_to_ensg = dict(zip(proteomics.index, proteomics.ENSG_ID))
index_to_hgnc = dict(zip(proteomics.index, proteomics.HGNC_ID))
proteomics.drop(columns = ['HGNC_ID'], inplace = True)

proteomics.drop(columns = ['reference'], inplace = True)#drop reference columns
# get unique column names, warning this gets rid fo tissue-specific information
proteomics.columns = list(range(proteomics.shape[1]))

# # median of samples across same tissue
# proteomics = proteomics.T
# proteomics['SAMPLE'] = proteomics.index
# proteomics.reset_index(inplace = True, drop = True)
# proteomics = proteomics.groupby('SAMPLE').median()
# proteomics.drop(index = ['reference'], inplace = True)
# proteomics = proteomics.T

# scale abundances by MW
for col in proteomics.columns:
    proteomics[col] = proteomics[col]*mw_
    
# get fractional biomass contribution
for col in proteomics.columns:
    proteomics[col] = proteomics[col]/proteomics[col].sum()
    
proteomics['MEDIAN'] = proteomics.median(axis = 1)
proteomics['MEDIAN'] = proteomics['MEDIAN']/proteomics['MEDIAN'].sum() # normalize to total of 1

proteomics.index = proteomics.index.map(index_to_hgnc)

In [278]:
proteomics['metabolic_machinery'] = False 
proteomics.loc[proteomics[proteomics.index.isin(metabolic_machinery)].index, 'metabolic_machinery'] = True

proteomics['expression_machinery'] = False 
proteomics.loc[proteomics[proteomics.index.isin(expression_machinery)].index, 'expression_machinery'] = True

proteomics['all_machinery'] = False 
proteomics.loc[proteomics[proteomics.index.isin(all_machinery)].index, 'all_machinery'] = True

median_frac = proteomics[proteomics.all_machinery]['MEDIAN'].sum()

0.12041534186261499

In [294]:
total_reactions_n = len(human_model.reactions)
no_gene_reactions_n = len([r for r in human_model.reactions if len(r.genes) == 0])
dummy_n = total_reactions_n/no_gene_reactions_n